# 5.2 Bayesian Linear & Logistic Regression

Regression becomes a distribution over plausible weights, so prediction carries both a mean and an uncertainty band.

Design matrices supply likelihood geometry, priors supply shrinkage, and posterior predictive distributions carry uncertainty into future predictions.

Save a copy to Drive to edit.

In [ ]:

import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(5202)


def bayes_linear_regression(X, y, sigma2, prior_var):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    prior_precision = np.eye(X.shape[1]) / prior_var
    precision = prior_precision + X.T @ X / sigma2
    covariance = np.linalg.inv(precision)
    mean = covariance @ X.T @ y / sigma2
    return mean, covariance


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def logistic_grid_posterior(X, y, grid):
    weights = np.array(list(itertools.product(grid, grid)), dtype=float)
    logits = X @ weights.T
    log_likelihood = y[:, None] * logits - np.logaddexp(0.0, logits)
    log_prior = -0.5 * np.sum(weights * weights, axis=1) / 4.0
    log_post = log_likelihood.sum(axis=0) + log_prior
    log_post = log_post - log_post.max()
    probs = np.exp(log_post)
    probs = probs / probs.sum()
    mean = probs @ weights
    covariance = (weights - mean).T @ ((weights - mean) * probs[:, None])
    return mean, covariance, weights, probs


def interval_width(x, covariance, sigma2, include_noise=True):
    variance = float(x @ covariance @ x)
    if include_noise:
        variance = variance + sigma2
    return 2.0 * 1.96 * math.sqrt(variance)


def marginal_error(mean, reference):
    return float(np.mean(np.abs(np.asarray(mean) - np.asarray(reference))))


## The concept, built once (D1)

Bayesian linear regression keeps the whole Gaussian posterior over weights rather than a single fitted coefficient.

$$p(w\mid X,y)\propto p(y\mid X,w)p(w),\quad \Sigma=(\Sigma_0^{-1}+X^\top X/\sigma^2)^{-1},\quad \mu=\Sigma X^\top y/\sigma^2$$

The reusable method computes posterior mean and covariance from the prior precision plus data precision.

In [ ]:

def d1_linear_example():
    X = np.array([[1.0], [2.0], [3.0]])
    y = np.array([1.2, 1.9, 3.2])
    mean, covariance = bayes_linear_regression(X, y, 0.25, 4.0)
    return X, y, mean, covariance


The lesson arithmetic is deliberately one-dimensional: precision $1/4+(1^2+2^2+3^2)/0.25=56.25$, posterior variance $0.017778$, weighted sum $58.4$, posterior mean $1.038222$, and predictive variance at $x_*=4$ of $0.534444$.

In [ ]:

X, y, mean, covariance = d1_linear_example()
precision = 1.0 / 4.0 + float((X.T @ X)[0, 0]) / 0.25
weighted_sum = float((X.T @ y)[0]) / 0.25
predictive_mean = 4.0 * float(mean[0])
predictive_variance = 0.25 + 16.0 * float(covariance[0, 0])
assert abs(precision - 56.25) < 1e-12
assert abs(float(covariance[0, 0]) - 0.017777777777777778) < 1e-12
assert abs(weighted_sum - 58.4) < 1e-12
assert abs(float(mean[0]) - 1.0382222222222223) < 1e-12
assert abs(predictive_mean - 4.152888888888889) < 1e-12
assert abs(predictive_variance - 0.5344444444444444) < 1e-12
print("posterior mean", mean)
print("posterior covariance", covariance)


## The dataset ladder

The ladder moves from exact one-weight Gaussian regression to four-point regression, a bimodal logistic grid, a correlated two-dimensional table, and an ill-conditioned higher-dimensional design.

In [ ]:

rungs = []

X = np.array([[1.0], [2.0]])
y = np.array([0.9, 2.2])
truth_mean, truth_cov = bayes_linear_regression(X, y, 0.16, 4.0)
rungs.append({"name": "D1 one weight", "mean": truth_mean, "cov": truth_cov, "ref": truth_mean, "sigma2": 0.16, "xstar": np.array([2.5])})

X = np.array([[1.0], [2.0], [3.0], [4.0]])
y = np.array([1.1, 1.8, 3.0, 4.1])
truth_mean, truth_cov = bayes_linear_regression(X, y, 0.25, 4.0)
rungs.append({"name": "D2 four points", "mean": truth_mean, "cov": truth_cov, "ref": truth_mean, "sigma2": 0.25, "xstar": np.array([4.5])})

X = np.array([[-2.0, 1.0], [-1.0, 1.0], [1.0, 1.0], [2.0, 1.0]])
y = np.array([0.0, 1.0, 0.0, 1.0])
grid = np.linspace(-5.0, 5.0, 121)
truth_mean, truth_cov, weights, probs = logistic_grid_posterior(X, y, grid)
rungs.append({"name": "D3 bimodal logistic", "mean": truth_mean, "cov": truth_cov, "ref": np.array([0.0, 0.0]), "sigma2": 1.0, "xstar": np.array([1.0, 1.0])})

X = np.array([[0.2, 0.1], [0.4, 0.3], [0.6, 0.65], [0.8, 0.85], [1.0, 1.05], [1.2, 1.25]])
y = np.array([0.3, 0.7, 1.0, 1.25, 1.7, 1.95])
X = (X - X.mean(axis=0)) / X.std(axis=0)
truth_mean, truth_cov = bayes_linear_regression(X, y, 0.09, 2.0)
rungs.append({"name": "D4 correlated 2-D", "mean": truth_mean, "cov": truth_cov, "ref": truth_mean, "sigma2": 0.09, "xstar": np.array([1.0, 1.0])})

n = 36
base = rng.normal(size=(n, 1))
X = np.hstack([base + 0.01 * rng.normal(size=(n, 1)) for _ in range(6)])
true_w = np.array([1.0, -0.8, 0.6, 0.0, 0.2, -0.1])
y = X @ true_w + 0.2 * rng.normal(size=n)
X = (X - X.mean(axis=0)) / X.std(axis=0)
truth_mean, truth_cov = bayes_linear_regression(X, y, 0.04, 1.0)
rungs.append({"name": "D5 ill-conditioned", "mean": truth_mean, "cov": truth_cov, "ref": true_w, "sigma2": 0.04, "xstar": np.ones(6)})

for rung in rungs:
    print(rung["name"], "weights", rung["mean"].shape[0], "mean sample", np.round(rung["mean"][:3], 3))


In [ ]:

errors = []
for rung in rungs:
    err = marginal_error(rung["mean"], rung["ref"])
    width = interval_width(rung["xstar"], rung["cov"], rung["sigma2"])
    errors.append(err)
    print(f"{rung['name']}: marginal_error={err:.6f}, predictive_interval_width={width:.3f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for i, rung in enumerate(rungs):
    x = np.arange(rung["mean"].shape[0])
    sd = np.sqrt(np.diag(rung["cov"]))
    axes[0, i].errorbar(x, rung["mean"], yerr=1.96 * sd, fmt="o", label="posterior")
    axes[0, i].scatter(x, rung["ref"], marker="x", label="reference")
    axes[0, i].set_title(rung["name"])
    axes[1, i].axis("off")
axes[0, 0].legend()
axes[1, 2].plot(range(1, 6), errors, marker="o")
axes[1, 2].set_xticks(range(1, 6))
axes[1, 2].set_xlabel("rung")
axes[1, 2].set_ylabel("marginal error")
axes[1, 2].set_title("error vs complexity")
fig.tight_layout()


## Pitfall on D5: confusing noise with parameter uncertainty

The predictive variance is $\sigma^2+x_*^\top\Sigma x_*$. Dropping the parameter-uncertainty term gives intervals that are too narrow on the ill-conditioned design.

In [ ]:

d5 = rungs[-1]
full_width = interval_width(d5["xstar"], d5["cov"], d5["sigma2"], include_noise=True)
noise_only_width = interval_width(d5["xstar"], d5["cov"], d5["sigma2"], include_noise=False)
print("noise plus parameter uncertainty width", full_width)
print("parameter-only width", noise_only_width)
print("missing variance term", d5["sigma2"])
assert full_width > noise_only_width


## Evaluate it + Practice

- Metric: mean absolute marginal error plus interval-width sanity checks against exact/grid references.
- No-skill baseline: use the prior, unary factors, or a single local conditional without global inference.
- Cheap sanity check: posterior covariance is positive definite and predictive variance grows away from observed geometry.
- Ablation: drop the prior precision or fail to standardize D5 features and compare unstable coefficients.
- Failure signals: singular design matrices, negative variances, or intervals narrower than observation noise allows.

Practice prompts:
1. Change the prior variance and observe shrinkage.

2. Try a denser logistic grid on D3.

3. Compare D5 intervals before and after feature scaling.